In [1]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import illustris_python as il

STAR = 4  # Tipo de partícula: estrellas
GAS = 0   # Tipo de partícula: gas

# ------------------ Unidades ------------------
def to_phys_length(x, a, h):   # kpc/h -> kpc
    return x * (a / h)

def to_phys_mass(m, h):        # (1e10 Msun/h) -> Msun
    return m * 1e10 / h

# ------------------ Elegir componente LOS ------------------
def los_component(vels, axis='Z'):
    axis = axis.upper()
    if axis == 'X': return vels[:,0]
    if axis == 'Y': return vels[:,1]
    return vels[:,2]  # 'Z' por defecto

# ------------------ PA e inclinación ------------------
def inertia_pa_incl(x_rel, y_rel, mass):
    I_xx = np.sum(mass * x_rel**2)
    I_yy = np.sum(mass * y_rel**2)
    I_xy = np.sum(mass * x_rel * y_rel)
    I = np.array([[I_xx, I_xy],[I_xy, I_yy]])
    eigvals, eigvecs = np.linalg.eigh(I)
    order = np.argsort(eigvals)
    eigvals = eigvals[order]; eigvecs = eigvecs[:,order]
    a_semi, b_semi = np.sqrt(eigvals[1] + 1e-30), np.sqrt(eigvals[0] + 1e-30)
    phi0 = np.degrees(np.arctan2(eigvecs[1,1], eigvecs[0,1])) % 180.0
    i_deg = np.degrees(np.arccos(np.clip(b_semi / a_semi, 0.0, 1.0)))
    return phi0, i_deg

# ------------------ Mapa de velocidad ------------------
def build_velocity_map(coords_kpc, vlos_kms, center_xy, rhalf_kpc, nbins=40, min_particles=9):
    x0, y0 = center_xy
    size = 6.0 * rhalf_kpc            # lado = 6 rhalf (radio 3 rhalf)
    dd = size / 2.0
    extent = [x0 - dd, x0 + dd, y0 - dd, y0 + dd]
    x = coords_kpc[:,0]; y = coords_kpc[:,1]

    xedges = np.linspace(extent[0], extent[1], nbins + 1)
    yedges = np.linspace(extent[2], extent[3], nbins + 1)
    ix = np.digitize(x, xedges) - 1
    iy = np.digitize(y, yedges) - 1
    valid = (ix >= 0) & (ix < nbins) & (iy >= 0) & (iy < nbins)

    df = pd.DataFrame({"ix": ix[valid], "iy": iy[valid], "v": vlos_kms[valid]})
    cnt = df.groupby(["iy","ix"])["v"].count()
    med = df.groupby(["iy","ix"])["v"].median()

    Vmap = np.full((nbins, nbins), np.nan)
    ok = cnt[cnt >= min_particles].index
    for (iyc, ixc) in ok:
        Vmap[iyc, ixc] = med.loc[(iyc, ixc)]

    pixel_kpc = (extent[1] - extent[0]) / nbins
    return Vmap, extent, pixel_kpc

def map_Rmax(Vmap, extent):
    ny, nx = Vmap.shape
    xv = np.linspace(extent[0], extent[1], nx)
    yv = np.linspace(extent[2], extent[3], ny)
    XX, YY = np.meshgrid(xv, yv)
    xc = 0.5*(extent[0] + extent[1]); yc = 0.5*(extent[2] + extent[3])
    R = np.hypot(XX - xc, YY - yc)
    M = np.isfinite(Vmap)
    return (np.nanmax(R[M]) if np.any(M) else 0.0)

# ------------------ Métricas del mapa (A y PCR) ------------------
def asymmetry_A(Vmap):
    if not np.any(np.isfinite(Vmap)): return np.nan
    V = Vmap.copy()
    VR = np.rot90(V, 2)
    M = np.isfinite(V) & np.isfinite(VR)
    den = 2.0 * np.nansum(np.abs(V[M]))
    if den == 0: return np.nan
    num = np.nansum(np.abs(V[M] + VR[M]))
    return num / den

def p_clear_rotation(Vmap, phi0_deg):
    if not np.any(np.isfinite(Vmap)): return np.nan
    ny, nx = Vmap.shape
    xv = np.linspace(-1, 1, nx)
    yv = np.linspace(-1, 1, ny)
    XX, YY = np.meshgrid(xv, yv)
    PA = (np.degrees(np.arctan2(YY, XX)) + 360.0) % 360.0
    pa0 = phi0_deg % 360.0
    dpa = np.abs(((PA - pa0 + 180.0) % 360.0) - 180.0)
    sel = (dpa <= 90.0) & np.isfinite(Vmap)
    if np.count_nonzero(sel) == 0: return np.nan
    Npos = np.count_nonzero(Vmap[sel] > 0)
    Nneg = np.count_nonzero(Vmap[sel] < 0)
    N = np.count_nonzero(sel)
    return np.abs(Npos - Nneg) / N

# ------------------ Masa estelar dentro de 2 rhalf ------------------
def stellar_mass_2rhalf(basePath, snapshot, subhalo_id, a, h):
    sh = il.groupcat.loadSingle(basePath, snapshot, subhaloID=int(subhalo_id))
    if 'SubhaloMassInRadType' in sh:
        return to_phys_mass(sh['SubhaloMassInRadType'][STAR], h)
    rhalf_kpc = to_phys_length(sh['SubhaloHalfmassRadType'][STAR], a, h)
    center_kpc = to_phys_length(sh['SubhaloPos'], a, h)
    parts = il.snapshot.loadSubhalo(basePath, snapshot, int(subhalo_id),
                                    partType=STAR, fields=['Coordinates','Masses'])
    
    # VERIFICACIÓN  para estrellas
    if parts is None or 'Coordinates' not in parts or len(parts['Coordinates']) == 0:
        return 0.0
        
    coords_kpc = to_phys_length(parts['Coordinates'], a, h)
    mstar = to_phys_mass(parts['Masses'], h)
    r = np.linalg.norm(coords_kpc - center_kpc, axis=1)
    return float(np.sum(mstar[r <= 2.0*rhalf_kpc]))

# ------------------ Masa GAS dentro de 2 rhalf ------------------
def gas_mass_2rhalf(basePath, snapshot, subhalo_id, a, h):
    sh = il.groupcat.loadSingle(basePath, snapshot, subhaloID=int(subhalo_id))
    rhalf_stars_kpc = to_phys_length(sh['SubhaloHalfmassRadType'][STAR], a, h)
    center_kpc = to_phys_length(sh['SubhaloPos'], a, h)
    
    gas_parts = il.snapshot.loadSubhalo(basePath, snapshot, int(subhalo_id),
                                        partType=GAS, fields=['Coordinates','Masses'])
    
    # --- CORRECCIÓN  KEYERROR ---
    if gas_parts is None or 'Coordinates' not in gas_parts or len(gas_parts['Coordinates']) == 0:
        return 0.0
    
    gas_coords = to_phys_length(gas_parts['Coordinates'], a, h)
    gas_masses = to_phys_mass(gas_parts['Masses'], h)
    
    dist = np.linalg.norm(gas_coords - center_kpc, axis=1)
    mask = dist <= (2.0 * rhalf_stars_kpc)
    
    return float(np.sum(gas_masses[mask]))

# ------------------ Filro de Aislamiento ------------------
def build_isolation_checker(header, subs, frac=0.10, r_iso_kpc=100.0): ## 10% de masa y 100 Kpc de distancia
    h = header['HubbleParam']; a = header['Time']
    Lbox = float(to_phys_length(header['BoxSize'], a, h))

    pos_phys = to_phys_length(subs['SubhaloPos'], a, h).astype(np.float64)
    mstar    = to_phys_mass(subs['SubhaloMassType'][:, STAR], h)

    good = (subs['SubhaloFlag'] == 1) & np.isfinite(mstar) & (mstar > 0.0)
    good_ids  = np.where(good)[0]
    pos_good  = pos_phys[good_ids]
    mstar_good = mstar[good_ids]

    pos_good = np.mod(pos_good, Lbox)
    eps = np.nextafter(0.0, 1.0)
    pos_good = np.minimum(pos_good, Lbox - eps)

    tree = cKDTree(pos_good, boxsize=Lbox)
    sid2idx = {int(sid): i for i, sid in enumerate(good_ids)}

    def is_isolated(sid):
        i = sid2idx.get(int(sid), None)
        if i is None:
            return False
        Mself = mstar_good[i]
        neigh = tree.query_ball_point(pos_good[i], r=r_iso_kpc)
        for j in neigh:
            if j == i:
                continue
            if mstar_good[j] >= frac * Mself:
                return False
        return True

    return is_isolated

# ------------------ Filtro Principal Completo ------------------
def select_galaxies_sim_params(basePath, snapshot=85, los='Z', nbins=40, min_particles=9, verbose=True):
    header = il.groupcat.loadHeader(basePath, snapshot)
    h = header['HubbleParam']; a = header['Time']
    print('el valor de a es : ', a)

    subs = il.groupcat.loadSubhalos(
        basePath, snapshot,
        fields=['SubhaloFlag','SubhaloPos','SubhaloVel','SubhaloHalfmassRadType','SubhaloMassType']
    )

    is_isolated = build_isolation_checker(header, subs, frac=0.10, r_iso_kpc=100.0)

    ids_all = np.arange(len(subs['SubhaloFlag']))
    ids = ids_all[subs['SubhaloFlag'] == 1]

    rows = []
    kept = 0
    for sid in ids:
        if not is_isolated(sid):
            continue

        M2 = stellar_mass_2rhalf(basePath, snapshot, sid, a, h)
        M3 = gas_mass_2rhalf(basePath, snapshot, sid, a, h)
        if not (M2 > 0.0 and 10.5 < np.log10(M2) < 11.5 and M3 >= 0.2 * M2): ## De Masa
            continue

        sh = il.groupcat.loadSingle(basePath, snapshot, subhaloID=int(sid))
        
        rhalf_kpc = to_phys_length(sh['SubhaloHalfmassRadType'][STAR], a, h)
        center    = to_phys_length(sh['SubhaloPos'], a, h)
        Vsys_vec  = sh['SubhaloVel']

        # --- COMPONENTE ESTELAR ---
        parts = il.snapshot.loadSubhalo(basePath, snapshot, int(sid),
                                        partType=STAR, fields=['Coordinates','Velocities','Masses'])
        
        if parts is None or 'Coordinates' not in parts or len(parts['Coordinates']) == 0:
            continue
        coords = to_phys_length(parts['Coordinates'], a, h)
        mstar  = to_phys_mass(parts['Masses'], h)
        vels   = parts['Velocities'] - Vsys_vec
        vlos   = los_component(vels, axis=los)

        rel = coords[:, :2] - center[:2]
        rproj = np.hypot(rel[:,0], rel[:,1])
        sel3 = (rproj <= 3.0 * rhalf_kpc)
        if np.count_nonzero(sel3) < 50:
            continue
        phi0_deg, i_deg = inertia_pa_incl(rel[sel3,0], rel[sel3,1], mstar[sel3])

        if (i_deg <= 10.0) or (i_deg >= 80.0):  ## Inclinacion 
            continue

        Vmap, extent, pixel_kpc = build_velocity_map(coords, vlos, center[:2], rhalf_kpc,
                                                     nbins=nbins, min_particles=min_particles)
        A = asymmetry_A(Vmap)
        if (not np.isfinite(A)) or (A >= 0.6):
            continue
        PCR = p_clear_rotation(Vmap, phi0_deg)
        if (not np.isfinite(PCR)) or (PCR <= 0.4):
            continue
        Rmax = map_Rmax(Vmap, extent)
        
        # --- BLOQUE DE GAS (CORREGIDO Y CONFIGURADO A 3*rhalf) ---
        parts_gas = il.snapshot.loadSubhalo(basePath, snapshot, int(sid),
                                            partType=GAS, fields=['Coordinates','Velocities','Masses'])
        
        # --- CORRECCIÓN CLAVE CONTRA EL KEYERROR EN EL BUCLE ---
        if parts_gas is None or 'Coordinates' not in parts_gas or len(parts_gas['Coordinates']) == 0:
            continue

        coords_gas = to_phys_length(parts_gas['Coordinates'], a, h)
        vels_gas   = parts_gas['Velocities'] - Vsys_vec
        vlos_gas   = los_component(vels_gas, axis=los)
        mgas       = to_phys_mass(parts_gas['Masses'], h)

        rel_gas = coords_gas[:, :2] - center[:2]
        rproj_gas = np.hypot(rel_gas[:,0], rel_gas[:,1])
        sel3_gas = (rproj_gas <= 3.0 * rhalf_kpc)
        if np.count_nonzero(sel3_gas) < 50:
            continue
            
        phi0_deg_gas, i_deg_gas = inertia_pa_incl(rel_gas[sel3_gas,0], rel_gas[sel3_gas,1], mgas[sel3_gas])
            
        Vmap_gas, _, _ = build_velocity_map(coords_gas, vlos_gas, center[:2], rhalf_kpc,
                                            nbins=nbins, min_particles=min_particles)
        
        A_gas = asymmetry_A(Vmap_gas)
        if (not np.isfinite(A_gas)) or (A_gas >= 0.6):
            continue
            
        PCR_gas = p_clear_rotation(Vmap_gas, phi0_deg_gas) 
        if (not np.isfinite(PCR_gas)) or (PCR_gas <= 0.4):
            continue
        
        # --- GUARDAR DATOS EN EL DATAFRAME ---
        rows.append(dict(
            SubhaloID=int(sid),
            isolated_100kpc=True,
            logM2rhalf=np.log10(M2),
            logMgas2rhalf=np.log10(M3),
            rhalf_kpc=rhalf_kpc,
            x0_kpc=center[0], y0_kpc=center[1], z0_kpc=center[2],
            Vsys_x_kms=Vsys_vec[0], Vsys_y_kms=Vsys_vec[1], Vsys_z_kms=Vsys_vec[2],
            i_deg=i_deg, phi0_deg=phi0_deg,
            i_deg_gas=i_deg_gas, phi0_deg_gas=phi0_deg_gas,
            pixel_kpc=pixel_kpc, Rmax_kpc=Rmax,
            asym_A_stars=A, PCR_stars=PCR,
            asym_A_gas=A_gas, PCR_gas=PCR_gas
        ))
        
        kept += 1
        print(f"✅ SubhaloID {sid} pasa: A_stars={A:.3f}, PCR_stars={PCR:.3f} | A_gas={A_gas:.3f}, PCR_gas={PCR_gas:.3f} (Total={kept})")
        if verbose and kept % 50 == 0:
            print(f"Acumulados (sin fit): {kept}")

    return pd.DataFrame(rows)

In [ ]:
# ------------------ uso ------------------
basePath = '/home/tnguser/sims.TNG/TNG100-1/output/'
snapshot = 99
#Filro completo
df = select_galaxies_sim_params(basePath, snapshot=snapshot, los='Z', nbins=40, min_particles=9, verbose=True)
print(len(df)); print(df.head()) 
#df.to_csv("tng100_selection_simparams.csv", index=False)

el valor de a es :  0.9999999999999998
✅ SubhaloID 41 pasa: A_stars=0.186, PCR_stars=0.967 | A_gas=0.190, PCR_gas=0.858 (Total=1)
✅ SubhaloID 17211 pasa: A_stars=0.114, PCR_stars=0.899 | A_gas=0.239, PCR_gas=0.571 (Total=2)
✅ SubhaloID 52630 pasa: A_stars=0.090, PCR_stars=0.978 | A_gas=0.092, PCR_gas=0.984 (Total=3)
